In [12]:
import pandas as pd

import numpy as np
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Detectar dispositivo — va aquí, una sola vez
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")


Dispositivo: cuda


In [2]:

# Carga del dataset
df = pd.read_csv('bike-sharing-demand/train.csv')

# Dimensiones generales
print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")


# Primeras 5 filas con todas las columnas
print("\nPrimeras 5 filas:")
print(df.head())

# Tipos de datos y valores nulos
print("\nInformación del dataset:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum())

Filas:    10886
Columnas: 12

Primeras 5 filas:
              datetime  season  holiday  workingday  weather  temp   atemp  \
0  2011-01-01 00:00:00       1        0           0        1  9.84  14.395   
1  2011-01-01 01:00:00       1        0           0        1  9.02  13.635   
2  2011-01-01 02:00:00       1        0           0        1  9.02  13.635   
3  2011-01-01 03:00:00       1        0           0        1  9.84  14.395   
4  2011-01-01 04:00:00       1        0           0        1  9.84  14.395   

   humidity  windspeed  casual  registered  count  
0        81        0.0       3          13     16  
1        80        0.0       8          32     40  
2        80        0.0       5          27     32  
3        75        0.0       3          10     13  
4        75        0.0       0           1      1  

Información del dataset:
datetime       object
season          int64
holiday         int64
workingday      int64
weather         int64
temp          float64
atemp        

In [3]:
# Convertir datetime una sola vez y guardarla como objeto fecha
dt = pd.to_datetime(df['datetime'])

# Extraer componentes temporales útiles
df['hour']      = dt.dt.hour        # hora del día (0-23)
df['month']     = dt.dt.month       # mes del año (1-12)
df['dayofweek'] = dt.dt.dayofweek   # día de la semana (0=lunes, 6=domingo)

# Eliminar columnas que no sirven
df = df.drop(columns=['datetime', 'casual', 'registered'])

# Verificar resultado
print(f"Columnas después del preprocesamiento: {df.shape[1]}")
print(f"\nColumnas actuales:")
print(df.columns.tolist())
print(f"\nPrimeras 5 filas:")
print(df.head())


Columnas después del preprocesamiento: 12

Columnas actuales:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'count', 'hour', 'month', 'dayofweek']

Primeras 5 filas:
   season  holiday  workingday  weather  temp   atemp  humidity  windspeed  \
0       1        0           0        1  9.84  14.395        81        0.0   
1       1        0           0        1  9.02  13.635        80        0.0   
2       1        0           0        1  9.02  13.635        80        0.0   
3       1        0           0        1  9.84  14.395        75        0.0   
4       1        0           0        1  9.84  14.395        75        0.0   

   count  hour  month  dayofweek  
0     16     0      1          5  
1     40     1      1          5  
2     32     2      1          5  
3     13     3      1          5  
4      1     4      1          5  


In [4]:

# Separar características (X) y variable objetivo (y)
y = df['count'].values

# X = features (todo excepto count)
X = df.drop(columns=['count']).values

m, n = X.shape

# Verificar dimensiones
print(f"Ejemplos de entrenamiento (m): {m}")
print(f"Features (n):                  {n}")
print(f"Dimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")

Ejemplos de entrenamiento (m): 10886
Features (n):                  11
Dimensiones de X: (10886, 11)
Dimensiones de y: (10886,)


In [6]:
# Separar features y variable objetivo
y = df['count'].values.astype(np.float32)  # float32 porque MSELoss lo requiere
X = df.drop(columns=['count']).values.astype(np.float32)

m, n = X.shape
print(f"Ejemplos (m): {m}")
print(f"Features (n): {n}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Normalización — igual que antes pero sin columna de unos
def featureNormalize(X):
    mu    = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1      # evitar división por cero
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

# mu y sigma se calculan ANTES del split
# en el siguiente bloque haremos el split correctamente
X_norm, mu, sigma = featureNormalize(X)

print(f"\nMedia (debe ser ~0):       {X_norm.mean():.6f}")
print(f"Desviación (debe ser ~1):  {X_norm.std():.6f}")


Ejemplos (m): 10886
Features (n): 11
X shape: (10886, 11)
y shape: (10886,)

Media (debe ser ~0):       -0.000000
Desviación (debe ser ~1):  1.000000


In [7]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

# Split DESPUÉS de normalizar
# Usamos los mismos mu y sigma para train y test
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]:,} filas")
print(f"Prueba:        {X_test.shape[0]:,} filas")

# Dataset personalizado
class BikeDataset(Dataset):
    def __init__(self, X, y):
        # X ya es float32, y ya es float32
        self.X = torch.tensor(X, dtype=torch.float32)
        # unsqueeze(1) convierte [m] → [m, 1]
        # porque MSELoss espera que y tenga forma [m, 1] igual que la predicción
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Crear datasets y dataloaders
train_dataset = BikeDataset(X_train, y_train)
test_dataset  = BikeDataset(X_test,  y_test)

dataloader = {
    'train': DataLoader(train_dataset, batch_size=256, shuffle=True,  pin_memory=True),
    'test':  DataLoader(test_dataset,  batch_size=256, shuffle=False, pin_memory=True)
}

print(f"\nBatches por época (train): {len(dataloader['train'])}")
print(f"Batches por época (test):  {len(dataloader['test'])}")

# Verificar forma de un batch
X_batch, y_batch = next(iter(dataloader['train']))
print(f"\nForma de un batch — X: {X_batch.shape}, y: {y_batch.shape}")


Entrenamiento: 8,708 filas
Prueba:        2,178 filas

Batches por época (train): 35
Batches por época (test):  9

Forma de un batch — X: torch.Size([256, 11]), y: torch.Size([256, 1])


In [13]:
class BikeNet(nn.Module):
    """
    Red neuronal para regresión — predice el número de bicicletas alquiladas.
    
    Diferencia clave con Opportunity:
    - Salida: 1 neurona (un número continuo) en vez de 5 (clases)
    - Sin Softmax — la salida es directamente el valor predicho
    - Loss: MSELoss en vez de CrossEntropyLoss
    """
    def __init__(self, n_inputs=9):
        super().__init__()
        
        self.red = nn.Sequential(
            # Capa 1: 9 → 64
            nn.Linear(n_inputs, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Capa 2: 64 → 32
            nn.Linear(64, 32),
            nn.ReLU(),
            
            # Salida: 32 → 1  ← un solo número
            nn.Linear(32, 1)
            # Sin ReLU al final — el modelo puede predecir cualquier valor
            # positivo o negativo libremente
        )
    
    def forward(self, x):
        return self.red(x)


# Instanciar modelo
model = BikeNet(n_inputs=n).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParámetros entrenables: {total_params:,}")

# Prueba rápida
with torch.no_grad():
    salida = model(X_batch.to(device))
print(f"\nForma de la salida: {salida.shape}")
print(f"Ejemplo de predicción cruda: {salida[0].item():.2f} bicicletas")


BikeNet(
  (red): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)

Parámetros entrenables: 2,881

Forma de la salida: torch.Size([256, 1])
Ejemplo de predicción cruda: 0.08 bicicletas


In [14]:
def fit(model, dataloader, epochs=100, checkpoint_path='./bike_checkpoint.pt'):
    
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # MSELoss — mide el error cuadrático medio entre predicción y valor real
    # es el equivalente de computeCostMulti de tu laboratorio anterior
    criterion = nn.MSELoss()
    
    historial = {
        'train_loss': [],
        'val_loss':   []
    }
    
    mejor_val_loss = float('inf')  # infinito — cualquier loss será mejor al inicio
    
    for epoch in range(1, epochs + 1):
        
        # ── ENTRENAMIENTO ──────────────────────────────
        model.train()
        train_loss = []
        
        bar = tqdm(dataloader['train'])
        for X_batch, y_batch in bar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            y_hat = model(X_batch)          # predicción: [256, 1]
            loss  = criterion(y_hat, y_batch)  # MSE entre predicción y real
            loss.backward()
            optimizer.step()
            
            train_loss.append(loss.item())
            bar.set_description(
                f"Epoch {epoch}/{epochs} [train] loss {np.mean(train_loss):.2f}"
            )
        
        # ── EVALUACIÓN ─────────────────────────────────
        model.eval()
        val_loss = []
        
        with torch.no_grad():
            bar = tqdm(dataloader['test'])
            for X_batch, y_batch in bar:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                y_hat = model(X_batch)
                loss  = criterion(y_hat, y_batch)
                val_loss.append(loss.item())
                
                bar.set_description(
                    f"Epoch {epoch}/{epochs} [val]  val_loss {np.mean(val_loss):.2f}"
                )
        
        epoch_train_loss = np.mean(train_loss)
        epoch_val_loss   = np.mean(val_loss)
        
        historial['train_loss'].append(epoch_train_loss)
        historial['val_loss'].append(epoch_val_loss)
        
        # Checkpoint — guarda cuando val_loss BAJA (al revés que accuracy)
        if epoch_val_loss < mejor_val_loss:
            mejor_val_loss = epoch_val_loss
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss':             epoch_val_loss,
            }, checkpoint_path)
            print(f"  ✓ Checkpoint guardado — epoch {epoch} val_loss {epoch_val_loss:.2f}")
        
        print(
            f"Epoch {epoch}/{epochs} | "
            f"train_loss {epoch_train_loss:.2f} | "
            f"val_loss {epoch_val_loss:.2f}"
        )
    
    return historial


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Cargar mejor modelo
checkpoint = torch.load('./bike_checkpoint.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Checkpoint cargado — epoch {checkpoint['epoch']} val_loss {checkpoint['val_loss']:.2f}")

# Evaluación completa
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in dataloader['test']:
        X_batch = X_batch.to(device)
        y_hat   = model(X_batch).cpu().numpy()
        all_preds.extend(y_hat.flatten())
        all_labels.extend(y_batch.numpy().flatten())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Métricas de regresión
mae  = mean_absolute_error(all_labels, all_preds)
rmse = np.sqrt(mean_squared_error(all_labels, all_preds))
print(f"\nMAE  (error absoluto medio):      {mae:.2f} bicicletas")
print(f"RMSE (error cuadrático medio):    {rmse:.2f} bicicletas")

# Predicciones concretas — equivalente a tu bloque de ejemplos
nombres_clases = ['Relaxing', 'Coffee time', 'Early morning', 'Cleanup', 'Sandwich time']

ejemplos = {
    'Verano mañana laboral':      [2, 0, 1, 1, 28, 31, 55, 10, 8,  7, 1],
    'Primavera tarde nublado':    [1, 0, 1, 2, 10, 12, 75, 20, 17, 4, 2],
    'Otoño feriado mediodía':     [3, 1, 0, 1, 22, 25, 60, 8,  14, 9, 3],
    'Invierno madrugada lluvia':  [4, 0, 0, 3, 5,  6,  90, 30, 2,  1, 6],
    'Verano hora pico laboral':   [2, 0, 1, 1, 30, 33, 45, 5,  18, 7, 4],
}
# Orden: season, holiday, workingday, weather, temp, atemp, humidity, windspeed, hour, month, dayofweek

print("\n" + "=" * 60)
print(f"{'Condición':>28} {'Predicción':>15}")
print("=" * 60)

model.eval()
with torch.no_grad():
    for condicion, valores in ejemplos.items():
        # 1. Convertir a array
        x = np.array(valores, dtype=np.float32)
        
        # 2. Normalizar con los mismos mu y sigma del entrenamiento
        x_norm = (x - mu) / sigma
        
        # 3. Convertir a tensor y agregar dimensión de batch [1, 9]
        x_tensor = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(device)
        
        # 4. Predecir
        pred = model(x_tensor).item()
        
        # 5. Clampear — no puede haber bicicletas negativas
        pred = max(0, pred)
        
        print(f"{condicion:>28} {pred:>12.0f} bicicletas")

print("=" * 60)

In [ ]:
# ── Gráfica 1: Convergencia del loss ────────────────────────
# equivalente a tu gráfica de convergencia del gradiente descendente
pyplot.figure(figsize=(10, 5))
pyplot.plot(historial['train_loss'], label='Train loss', color='steelblue')
pyplot.plot(historial['val_loss'],   label='Val loss',   color='coral')
pyplot.xlabel('Época')
pyplot.ylabel('MSE Loss')
pyplot.title('Convergencia del entrenamiento')
pyplot.legend()
pyplot.grid(True)
pyplot.show()

# ── Gráfica 2: Predicción vs Real ───────────────────────────
pyplot.figure(figsize=(8, 6))
pyplot.scatter(all_labels, all_preds, alpha=0.3, color='steelblue', s=10)
pyplot.plot(
    [all_labels.min(), all_labels.max()],
    [all_labels.min(), all_labels.max()],
    color='coral', lw=2, label='Predicción perfecta'
)
pyplot.xlabel('Valores reales')
pyplot.ylabel('Valores predichos')
pyplot.title('Predicción vs Real — conjunto de prueba')
pyplot.legend()
pyplot.grid(True)
pyplot.show()

# ── Gráfica 3: Comparación con modelo anterior ──────────────
# Reemplaza con tus valores reales del laboratorio anterior
mae_anterior  = 0.0    # <-- pega tu MAE del laboratorio anterior
rmse_anterior = 0.0    # <-- pega tu RMSE del laboratorio anterior

modelos  = ['Regresión Lineal\n(Lab anterior)', 'Red Neuronal\n(PyTorch)']
maes     = [mae_anterior,  mae]
rmses    = [rmse_anterior, rmse]

x     = np.arange(len(modelos))
width = 0.35

fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(12, 5))

# MAE
barras = ax1.bar(x, maes, width, color=['steelblue', 'coral'])
for barra in barras:
    altura = barra.get_height()
    ax1.text(barra.get_x() + barra.get_width()/2, altura + 0.5,
             f'{altura:.2f}', ha='center', va='bottom')
ax1.set_ylabel('MAE (bicicletas)')
ax1.set_title('Error absoluto medio')
ax1.set_xticks(x)
ax1.set_xticklabels(modelos)
ax1.grid(axis='y')

# RMSE
barras = ax2.bar(x, rmses, width, color=['steelblue', 'coral'])
for barra in barras:
    altura = barra.get_height()
    ax2.text(barra.get_x() + barra.get_width()/2, altura + 0.5,
             f'{altura:.2f}', ha='center', va='bottom')
ax2.set_ylabel('RMSE (bicicletas)')
ax2.set_title('Error cuadrático medio')
ax2.set_xticks(x)
ax2.set_xticklabels(modelos)
ax2.grid(axis='y')

pyplot.tight_layout()
pyplot.show()

# ── Resumen final ────────────────────────────────────────────
print("=" * 55)
print("         RESUMEN DE RESULTADOS")
print("=" * 55)
print(f"\nModelo anterior — Regresión Lineal:")
print(f"  MAE:  {mae_anterior:.2f} bicicletas")
print(f"  RMSE: {rmse_anterior:.2f} bicicletas")
print(f"\nModelo nuevo — Red Neuronal PyTorch:")
print(f"  Arquitectura:  {n} → 64 → 32 → 1")
print(f"  Épocas:        100")
print(f"  Batch size:    256")
print(f"  Optimizador:   Adam (lr=1e-3)")
print(f"  MAE:           {mae:.2f} bicicletas")
print(f"  RMSE:          {rmse:.2f} bicicletas")
print(f"\nCheckpoint guardado en: ./bike_checkpoint.pt")
print(f"  Mejor época:   {checkpoint['epoch']}")
print(f"  Mejor val_loss:{checkpoint['val_loss']:.2f}")
print("=" * 55)
